# CO₂ Storage Capacity Assessment

This notebook runs a probabilistic static storage-capacity assessment using:

$$SC = GRV \times (N/G) \times \phi \times \rho_{CO_2} \times S_{eff}$$

Change the values in the **Editable inputs** cell, then choose **Runtime → Run all**. The example values represent the Rødby Bunter Sandstone assessment.

In [ ]:
#@title Install dependencies { display-mode: "form" }
# Install the latest package and plotting tools from GitHub.
%pip install -q "git+https://github.com/AnaSoles/ggg-co2-storage-eval.git" matplotlib pandas

In [ ]:
#@title Import libraries { display-mode: "form" }
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from storageeval import Distribution, StorageSite, simulate

plt.style.use("seaborn-v0_8-whitegrid")

## Editable inputs

Enter minimum, most likely, and maximum values. Fractions such as porosity must be decimals: `0.23` means 23%. The Rødby values come from GEUS Report 2024/18, Table 8.4.1.

**Source corrections:** Table 8.2.1 prints the maximum GRV as 23.9 km³, while Table 8.4.1 gives 33.85 km³; the latter agrees with the stated ±20% method around the 28.21 km³ mode. Table 8.4.1 also prints the maximum CO₂ density as 764.0 kg/m³, but Section 8.2.4 says the maximum is 10% above the 603.6 kg/m³ mode. The internally consistent values, 33.85 km³ and 663.96 kg/m³, are used below because they reproduce the report's capacity results.

In [ ]:
site_name = "Rødby – Bunter Sandstone"
iterations = 100_000
random_seed = 42

#                       minimum, most likely, maximum
grv_km3 =               (22.57, 28.21, 33.85)
net_to_gross =          (0.20,  0.25,  0.30)
porosity =              (0.184, 0.23,  0.276)
co2_density_kg_m3 =     (573.4, 603.6, 663.96)  # Table prints 764.0; Section 8.2.4 implies 663.96
storage_efficiency =    (0.05,  0.10,  0.20)

## Input parameter table

This table is generated from the editable values above, so it updates automatically when you change an input. `Description / interpretation`, `Input type`, and `Source / basis` consolidate the meaning and provenance of every Rødby parameter in one place.

In [ ]:
#@title Show input parameter table { display-mode: "form" }
input_table = pd.DataFrame([
    ["Gross rock volume (GRV)", "km³", "PERT", *grv_km3, "Seismic-derived estimate", "Seismic-mapped Bunter Sandstone reservoir volume within the Rødby structural closure. The range comes from the structural volumetric model; 33.85 km³ is the value consistent with Table 8.4.1 and the stated ±20% method.", "GEUS Report 2024/18, Sections 8.1–8.2.1 and Table 8.4.1"],
    ["Net-to-gross (N/G)", "fraction", "PERT", *net_to_gross, "Geological / well-derived estimate", "Fraction of the gross Bunter Sandstone interval interpreted as effective reservoir sandstone, based on Rødby-1, Rødby-2 and surrounding wells.", "GEUS Report 2024/18, Section 8.2.2 and Table 8.4.1"],
    ["Porosity (φ)", "fraction", "PERT", *porosity, "Petrophysical estimate", "Fraction of reservoir bulk volume occupied by pore space. The 23% mode comes from well-based petrophysical interpretation, with approximately ±20% uncertainty for spatial and vertical variability.", "Rødby-1, Rødby-2 and surrounding wells; GEUS Report 2024/18, Sections 7.1 and 8.2.3"],
    ["In-situ CO₂ density", "kg/m³", "PERT", *co2_density_kg_m3, "Thermodynamic estimate", "Calculated CO₂ density at reservoir conditions, not a direct well measurement. GEUS applies about −5%/+10% around 603.6 kg/m³; the maximum is therefore 603.6 × 1.10 = 663.96 kg/m³.", "GEUS Report 2024/18, Section 8.2.4; CO₂ properties based on Span & Wagner (1996)"],
    ["Storage efficiency", "fraction", "PERT", *storage_efficiency, "Literature-informed, site-specific assumption", "Assumed fraction of available pore volume effectively occupied by stored CO₂. It is a screening assumption, not a direct Rødby measurement. The 10% mode reflects generally good Bunter Sandstone properties; revise it when dynamic simulation or more site-specific data become available.", "GEUS Report 2024/18, Sections 5.5 and 8.3; Goodman et al. (2011), Gorecki et al. (2009), Wang et al. (2013)"],
], columns=["Parameter", "Unit", "Distribution", "Minimum", "Mode", "Maximum", "Input type", "Description / interpretation", "Source / basis"])
input_table

## Rødby input-data provenance

The probabilistic inputs are based on Abramovitz et al. (2024), GEUS Report 2024/18, and use independent PERT distributions in the static volumetric equation:

$$SC = GRV \times (N/G) \times \phi \times \rho_{CO_2} \times S_{eff}$$

The parameter descriptions, input classifications, uncertainty rationale, and source basis are consolidated in the **Input parameter table** above.

### References

- Abramovitz, T., Vosgerau, H., Gregersen, U., et al. (2024). *CCS2022-2024 WP1: The Rødby structure – Seismic data and interpretation to mature potential geological storage of CO₂*. GEUS Report 2024/18. [https://doi.org/10.22008/gpub/34739](https://doi.org/10.22008/gpub/34739)
- Goodman, A., Hakala, J. A., Bromhal, G., Deel, D., Rodosta, T., Frailey, S., et al. (2011). *U.S. DOE methodology for the development of geologic storage potential for carbon dioxide at the national and regional scale*. *International Journal of Greenhouse Gas Control, 5*(4), 952–965. [https://doi.org/10.1016/j.ijggc.2011.03.010](https://doi.org/10.1016/j.ijggc.2011.03.010)
- Gorecki, C. D., Sorensen, J. A., Bremer, J. M., et al. (2009). *Development of Storage Coefficients for Carbon Dioxide Storage in Deep Saline Formations*. IEA Greenhouse Gas R&D Programme.
- Span, R. & Wagner, W. (1996). *A New Equation of State for Carbon Dioxide Covering the Fluid Region from the Triple-Point Temperature to 1100 K at Pressures up to 800 MPa*. *Journal of Physical and Chemical Reference Data, 25*(6), 1509–1596. [https://doi.org/10.1063/1.555991](https://doi.org/10.1063/1.555991)
- Wang, Y., Zhang, K. & Wu, N. (2013). *Numerical Investigation of the Storage Efficiency Factor for CO₂ Geological Sequestration in Saline Formations*. *Energy Procedia, 37*, 5267–5274. [https://doi.org/10.1016/j.egypro.2013.06.443](https://doi.org/10.1016/j.egypro.2013.06.443)

In [ ]:
#@title Run Monte Carlo simulation and compare with GEUS { display-mode: "form" }
site = StorageSite(
    name=site_name,
    grv=Distribution.pert(*grv_km3),
    net_to_gross=Distribution.pert(*net_to_gross),
    porosity=Distribution.pert(*porosity),
    co2_density=Distribution.pert(*co2_density_kg_m3),
    storage_efficiency=Distribution.pert(*storage_efficiency),
)

result = simulate(site, iterations=iterations, seed=random_seed)
summary = result.summary()
report_values = {"p90_mt": 68.83, "p50_mt": 103.90, "p10_mt": 148.78, "mean_mt": 107.04}
result_rows = [("P90 (conservative)", "p90_mt"), ("P50 (median)", "p50_mt"), ("P10 (upside)", "p10_mt"), ("Mean", "mean_mt")]
comparison = pd.DataFrame({
    "Notebook (Mt CO₂)": [summary[key] for _, key in result_rows],
    "GEUS Table 8.5.1 (Mt CO₂)": [report_values[key] for _, key in result_rows],
}, index=[label for label, _ in result_rows])
comparison["Difference (Mt CO₂)"] = comparison["Notebook (Mt CO₂)"] - comparison["GEUS Table 8.5.1 (Mt CO₂)"]
comparison.round(2)

The notebook and GEUS values should be very close, but not numerically identical. Both use the same static volumetric equation and independent PERT inputs. Small differences are expected because the report does not state its number of Monte Carlo iterations, random seed, or exact software implementation of PERT.

## Input uncertainty distributions

In [ ]:
#@title Show input uncertainty distributions { display-mode: "form" }
labels = {
    "grv_km3": "GRV (km³)",
    "net_to_gross": "Net-to-gross",
    "porosity": "Porosity",
    "co2_density_kg_m3": "CO₂ density (kg/m³)",
    "storage_efficiency": "Storage efficiency",
}
input_ranges = {
    "grv_km3": grv_km3,
    "net_to_gross": net_to_gross,
    "porosity": porosity,
    "co2_density_kg_m3": co2_density_kg_m3,
    "storage_efficiency": storage_efficiency,
}
line_styles = [
    ("Minimum", "#1f77b4", ":"),
    ("Mode", "#2ca02c", "--"),
    ("Mean", "#ff7f0e", "-"),
    ("Maximum", "#d62728", ":"),
]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, values) in zip(axes.flat, result.inputs.items()):
    minimum, mode, maximum = input_ranges[name]
    mean = float(np.mean(values))
    reference_values = [minimum, mode, mean, maximum]
    ax.hist(values, bins=45, color="#b9d7f0", edgecolor="white", alpha=0.9)
    for (line_name, color, linestyle), reference in zip(line_styles, reference_values):
        ax.axvline(
            reference,
            color=color,
            linestyle=linestyle,
            linewidth=1.6,
            label=f"{line_name}: {reference:.4g}",
        )
    ax.set_title(labels[name])
    ax.set_ylabel("Simulations")
    ax.legend(fontsize=8, frameon=True, loc="upper right")
axes.flat[-1].axis("off")
fig.suptitle(f"{site_name} – input uncertainty distributions and values used", fontsize=15)
fig.tight_layout()
plt.show()


## Storage-capacity probability distribution

Grey bars show the simulated Rødby capacities, the orange line is a fitted lognormal probability density, and the red curve is cumulative exceedance probability. The P90, P50 and P10 markers use the same Monte Carlo result reported in the comparison table.


In [ ]:
#@title Show capacity histogram with fitted density and cumulative curve { display-mode: "form" }
def plot_capacity_distribution(ax_density, result_name, result):
    values = np.asarray(result.capacity_mt, dtype=float)
    summary = result.summary()

    ax_density.hist(
        values,
        bins=50,
        density=True,
        color="#d9d9d9",
        edgecolor="white",
        linewidth=0.5,
        label="Simulated capacity",
    )

    log_values = np.log(values)
    mu = float(np.mean(log_values))
    sigma = float(np.std(log_values, ddof=1))
    x = np.linspace(float(np.min(values)), float(np.max(values)), 500)
    fitted_density = np.exp(
        -0.5 * ((np.log(x) - mu) / sigma) ** 2
    ) / (x * sigma * np.sqrt(2 * np.pi))
    ax_density.plot(x, fitted_density, color="#f39c12", linewidth=2.0, label="Fitted density")

    ax_cumulative = ax_density.twinx()
    capacity_sorted = np.sort(values)
    exceedance_pct = (
        1 - np.arange(1, capacity_sorted.size + 1) / (capacity_sorted.size + 1)
    ) * 100
    ax_cumulative.plot(
        capacity_sorted,
        exceedance_pct,
        color="#e31a1c",
        linewidth=1.8,
        label="Cumulative exceedance",
    )

    markers = [
        ("P90", "p90_mt", 90, "#b2182b"),
        ("P50", "p50_mt", 50, "#ff8c42"),
        ("P10", "p10_mt", 10, "#b2182b"),
    ]
    for marker_label, key, probability, color in markers:
        marker_value = summary[key]
        ax_density.axvline(
            marker_value,
            color=color,
            linestyle="--",
            linewidth=1.1,
            alpha=0.9,
        )
        ax_cumulative.plot(marker_value, probability, "o", color=color, markersize=4)
        vertical_offset = 8 if marker_label != "P10" else -14
        ax_cumulative.annotate(
            f"{marker_label}: {marker_value:.1f} Mt",
            (marker_value, probability),
            xytext=(5, vertical_offset),
            textcoords="offset points",
            color=color,
            fontsize=9,
        )

    ax_density.text(
        0.98,
        0.96,
        f"Mean: {summary['mean_mt']:.1f} Mt\nSD: {np.std(values, ddof=1):.1f} Mt",
        transform=ax_density.transAxes,
        ha="right",
        va="top",
        color="#b56500",
        fontsize=9,
    )
    ax_density.set_title(result_name)
    ax_density.set_xlabel("Storage capacity (Mt CO₂)")
    ax_density.set_ylabel("Probability density")
    ax_cumulative.set_ylabel("Exceedance probability (%)", color="#e31a1c")
    ax_cumulative.set_ylim(0, 100)
    ax_cumulative.tick_params(axis="y", colors="#e31a1c")

    density_handles, density_labels = ax_density.get_legend_handles_labels()
    cumulative_handles, cumulative_labels = ax_cumulative.get_legend_handles_labels()
    ax_density.legend(
        density_handles + cumulative_handles,
        density_labels + cumulative_labels,
        loc="upper center",
        fontsize=8,
    )

fig, ax = plt.subplots(figsize=(11, 7))
plot_capacity_distribution(ax, site_name, result)
fig.tight_layout()
plt.show()


## Exceedance curve

P90 is the capacity that has a 90% probability of being exceeded; P10 is the upside estimate.

In [ ]:
#@title Show exceedance curve { display-mode: "form" }
fig, ax = result.plot_exceedance()
plt.show()

## Linear capacity confidence ranges

The colored bar summarizes conservative, central, and upside capacity ranges. These are probabilistic static capacity estimates, not booked reserves.

In [ ]:
#@title Show linear capacity confidence ranges { display-mode: "form" }
fig, ax = result.plot_capacity_ranges()
plt.show()

## One-at-a-time capacity tornado

For each bar, one input moves from its P90 input value (10th sample percentile) to its P10 input value (90th sample percentile), while the other four inputs remain at their sampled means. The labels show the resulting low and high storage capacities. This measures direct capacity impact and complements—not replaces—the Spearman rank chart below.


In [ ]:
#@title Show one-at-a-time capacity tornado { display-mode: "form" }
def capacity_from_inputs(parameter_values):
    return float(np.prod(list(parameter_values.values())))

sample_means = {
    name: float(np.mean(samples))
    for name, samples in result.inputs.items()
}
baseline_capacity = capacity_from_inputs(sample_means)
tornado_rows = []

for name, samples in result.inputs.items():
    low_inputs = sample_means.copy()
    high_inputs = sample_means.copy()
    low_inputs[name] = float(np.quantile(samples, 0.10))
    high_inputs[name] = float(np.quantile(samples, 0.90))
    tornado_rows.append(
        (
            labels[name],
            capacity_from_inputs(low_inputs),
            capacity_from_inputs(high_inputs),
        )
    )

ordered_rows = sorted(tornado_rows, key=lambda row: row[2] - row[1])
parameter_names = [row[0] for row in ordered_rows]
low_capacities = np.asarray([row[1] for row in ordered_rows])
high_capacities = np.asarray([row[2] for row in ordered_rows])
bar_widths = high_capacities - low_capacities

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(
    parameter_names,
    bar_widths,
    left=low_capacities,
    color="#5b9bd5",
    edgecolor="#2f5597",
    alpha=0.9,
)
ax.axvline(
    baseline_capacity,
    color="#c00000",
    linestyle="--",
    linewidth=1.4,
    label=f"Base: {baseline_capacity:.1f} Mt",
)

chart_range = max(
    float(np.max(high_capacities) - np.min(low_capacities)),
    1e-12,
)
padding = 0.10 * chart_range
ax.set_xlim(
    float(np.min(low_capacities) - padding),
    float(np.max(high_capacities) + padding),
)

for bar, low, high, width in zip(
    bars,
    low_capacities,
    high_capacities,
    bar_widths,
):
    y = bar.get_y() + bar.get_height() / 2
    if width >= 0.16 * chart_range:
        ax.text(
            low + 0.03 * width,
            y,
            f"{low:.1f}",
            ha="left",
            va="center",
            color="white",
            fontsize=9,
            fontweight="bold",
        )
        ax.text(
            high - 0.03 * width,
            y,
            f"{high:.1f}",
            ha="right",
            va="center",
            color="white",
            fontsize=9,
            fontweight="bold",
        )
    else:
        ax.annotate(
            f"{low:.1f}",
            (low, y),
            xytext=(-4, 0),
            textcoords="offset points",
            ha="right",
            va="center",
            color="#1f1f1f",
            fontsize=8,
            fontweight="bold",
        )
        ax.annotate(
            f"{high:.1f}",
            (high, y),
            xytext=(4, 0),
            textcoords="offset points",
            ha="left",
            va="center",
            color="#1f1f1f",
            fontsize=8,
            fontweight="bold",
        )

ax.set_title(f"{site_name} – one-at-a-time capacity sensitivity")
ax.set_xlabel("Storage capacity (Mt CO₂)")
ax.legend(fontsize=9, loc="lower right")
fig.tight_layout()
plt.show()


## Spearman rank sensitivity

This second chart retains the original global sensitivity view. Longer bars identify inputs with the strongest monotonic association with simulated capacity; the printed value is the Spearman rank-correlation coefficient.


In [ ]:
#@title Show Spearman rank sensitivity { display-mode: "form" }
fig, ax = result.plot_sensitivity()
fig.set_size_inches(10, 6)

readable_tick_labels = [labels.get(tick.get_text(), tick.get_text()) for tick in ax.get_yticklabels()]
ax.set_yticks(ax.get_yticks(), labels=readable_tick_labels)

for bar in ax.patches:
    value = float(bar.get_width())
    y = bar.get_y() + bar.get_height() / 2
    if abs(value) >= 0.14:
        inset = 0.025 if value >= 0 else -0.025
        ax.text(
            value - inset,
            y,
            f"{value:.2f}",
            ha="right" if value >= 0 else "left",
            va="center",
            color="white",
            fontsize=9,
            fontweight="bold",
        )
    else:
        offset = 5 if value >= 0 else -5
        ax.annotate(
            f"{value:.2f}",
            (value, y),
            xytext=(offset, 0),
            textcoords="offset points",
            ha="left" if value >= 0 else "right",
            va="center",
            color="#1f1f1f",
            fontsize=9,
            fontweight="bold",
        )

plt.show()


## Important limitation

This is a static volumetric screening assessment. It does not yet represent pressure constraints, injectivity, plume migration, dynamic reservoir simulation, or economics.